# Lake House

In [8]:
import pandas as pd
import numpy as np
from deltalake.writer import write_deltalake
from minio import Minio
from pathlib import Path

## Generate data

In [2]:
data_dict = {
    'name': [''.join(np.random.choice(list('abcdefghijklmnopqrstuvwxyz'), size=8)) for _ in range(100)],
    'price': np.random.random(100).tolist(),
    'quantity': np.random.randint(1, 10_000, size=100).tolist(),
}

df = pd.DataFrame(data_dict)
df.head()

,name,price,quantity
0,atqtzlxa,0.781860,9545
1,qacnyqjz,0.896496,3062
2,kcvtmzfi,0.077554,3584
3,whenlfsy,0.931876,5288
4,lhisasdd,0.152970,670


In [3]:
split_idx = np.array_split(df.index, 3)
df_part_1, df_part_2, df_part_3 = [df.loc[idx].reset_index(drop=True) for idx in split_idx]
df_part_1, df_part_2, df_part_3

(        name     price  quantity
 0   atqtzlxa  0.781860      9545
 1   qacnyqjz  0.896496      3062
 2   kcvtmzfi  0.077554      3584
 3   whenlfsy  0.931876      5288
 4   lhisasdd  0.152970       670
 5   vaypncdp  0.211383        26
 6   vwjtkwky  0.036366      6333
 7   dxpdvrqc  0.437544      9256
 8   pxkkpmtb  0.330570      4312
 9   letbcnrl  0.478818      3194
 10  fpjauskm  0.090298      7831
 11  kbavyhyx  0.012472      3208
 12  eynlqbni  0.987240      6785
 13  kmnhrrup  0.478596      2382
 14  upscznvq  0.887012      2244
 15  rjubkkiq  0.772764      8616
 16  xcmgvwub  0.246290      1161
 17  pflcigeo  0.225444      4702
 18  ktawyhcm  0.102394      7129
 19  vhgssyyn  0.709400      6394
 20  ybkrnvsw  0.376399      7887
 21  jxigpbue  0.108172      3279
 22  qnzdcoau  0.108092      3272
 23  clifwyem  0.667976      6671
 24  murwzoyr  0.011970      1917
 25  dmfwduld  0.633387      6225
 26  okggqrrc  0.716081      3558
 27  yeyjlnjn  0.104527      8819
 28  gbupjdzi 

In [5]:
all_df:list[pd.DataFrame] = [df_part_1, df_part_2, df_part_3]
for idx, part in enumerate(all_df):
    write_deltalake(f'../../data/test/lh/part_{idx}', part)

## Push data to MinIO

In [6]:
client = Minio(
    endpoint='localhost:9099',
    access_key='minio_access_key',
    secret_key='minio_secret_key',
    secure=False,
)

In [7]:
bucket_name = 'store-lh'
found = client.bucket_exists(bucket_name=bucket_name)
if not found:
    client.make_bucket(bucket_name=bucket_name)
else:
    print(f'Bucket {bucket_name} already exists, skip creating!')

Bucket store-lh already exists, skip creating!


In [9]:
data_path = Path('../../data/test/lh')

for fp in list(data_path.rglob('*.parquet')) + list(data_path.rglob('*.json')):
    print(f"Uploading {fp}")
    client.fput_object(
        bucket_name=bucket_name,
        object_name=fp._str[len('../../data/test/lh/'):],
        file_path=fp
    )

Uploading ../../data/test/lh/part_2/part-00000-ea87791a-c43a-49f8-9c51-535080da6e8a-c000.snappy.parquet
Uploading ../../data/test/lh/part_1/part-00000-2184c646-bd9d-4bec-91ac-71f271bd3c94-c000.snappy.parquet
Uploading ../../data/test/lh/part_0/part-00000-9219f541-fa8c-480b-8166-fb2f2ba086ba-c000.snappy.parquet
Uploading ../../data/test/lh/part_2/_delta_log/00000000000000000000.json
Uploading ../../data/test/lh/part_1/_delta_log/00000000000000000000.json
Uploading ../../data/test/lh/part_0/_delta_log/00000000000000000000.json


## Query from Trino

- Create schema:

    ```sql
    CREATE SCHEMA IF NOT EXISTS lakehouse.store
            WITH (location = 's3://store/');
    ```

- Register a schema for existing tables (repeat 3 times)

    ```sql
    CALL lakehouse.system.register_table(
        schema_name => 'store',
        table_name => 'product_0',
        table_location => 's3://store/product/part_0'
    )
    ```

    - By doing so, `Trino` automatically figures it out the suitable schema for these table and the corresponding log files $\to$ Can query from now on on `Trino`

- Apply a view for these tables for a better look:

    ```sql
    CREATE OR REPLACE VIEW lakehouse.store.product AS
        SELECT * FROM lakehouse.store.product_0
        UNION ALL
        SELECT * FROM lakehouse.store.product_1
        UNION ALL
        SELECT * FROM lakehouse.store.product_2;
    ```

- Ever since we use `DeltaLake`, every modification will be recorded so we can roll back easily

    ```sql
    INSERT INTO lakehouse.store.product_2 ("date", "name", "price", "quantity")
        VALUES (CURRENT_TIMESTAMP, 'dummy_product', 0.0, 0);
    ```

    - Make it 4 times

    - Then, new data + log file appearing in the bucket

- Roll back:
    - Đầu tiên, cần xác định version dữ liệu cần lấy ra thông qua lệnh

        ```bash
        use lakehouse.store;
        select version, timestamp, operation from "product_2$history";
        ```

        - Output:

            ```txt
            version  |          timestamp          | operation 
            ---------+-----------------------------+-----------
                   0 | 2026-04-21 03:49:08.480 UTC | WRITE     
                   1 | 2026-04-21 12:22:17.519 UTC | WRITE     
                   2 | 2026-04-21 12:22:23.375 UTC | WRITE     
                   3 | 2026-04-21 12:22:28.347 UTC | WRITE
            ```

    - Sau đó note lại version cần lấy và truy vấn theo 3 dạng sau:
        - TH1: Data đủ nhỏ để giữ trên RAM. Query trực tiếp `Trino` từ source $\to$ Trả về `Dataframe`
        - TH2: Data lớn hơn RAM. Lúc này cần lưu ra file parquet và dùng ML lib để train từ file đó. File output parquet có thể export ra local hoặc `MinIO`